# Stage 3 — Vibration-Based Fault Detection (NASA IMS Bearing Dataset)

Unlike Stages 1-2 (RUL regression on C-MAPSS operational sensors), this stage
works with **raw high-frequency vibration signals** and frames the problem as
**fault-stage classification**: Normal -> Degrading -> Near-Failure.

Two approaches, compared head-to-head:
- Classical: hand-engineered time/frequency features -> Random Forest
- Deep learning: spectrogram images -> CNN

**Enable a GPU**: Runtime -> Change runtime type -> T4 GPU (helps the CNN step).

## 1. Install dependencies

In [ ]:
!pip install -q scikit-learn scipy

## 1b. Cache check (Google Drive) — run this before downloading anything

Downloading and processing the raw IMS dataset (~1 GB, plus reading 2,156
individual files twice — once for features, once for spectrograms) is slow
and has to repeat every time a fresh Colab runtime starts. This cell checks
Google Drive for a cached, already-processed version from a previous run.

**First time ever running this notebook:** nothing will be cached yet —
just continue to Section 2 as normal. Save-to-cache cells later (after
Section 5 and Section 11) will populate the cache for next time.

**Every time after that:** if the cache is found, this loads `feature_df`
and the full spectrogram array directly — you can then **skip Sections 2
through 5 and Section 10-11 entirely** and jump straight to Section 6.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import numpy as np
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/stage3_vibration_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

CACHE_FEATURE_DF = os.path.join(CACHE_DIR, "feature_df.csv")
CACHE_SPECTROGRAMS = os.path.join(CACHE_DIR, "all_spectrograms.npy")

CACHE_AVAILABLE = os.path.exists(CACHE_FEATURE_DF)
SPECTROGRAM_CACHE_AVAILABLE = os.path.exists(CACHE_SPECTROGRAMS)

if CACHE_AVAILABLE:
    feature_df = pd.read_csv(CACHE_FEATURE_DF)
    print(f"Loaded cached feature_df from Drive: {feature_df.shape}")
    print("You can SKIP Sections 2-5 now and go straight to Section 6.")
else:
    print("No cached feature_df found yet — continue to Section 2 as normal.")

if SPECTROGRAM_CACHE_AVAILABLE:
    X_spec = np.load(CACHE_SPECTROGRAMS)
    print(f"Loaded cached spectrograms from Drive: {X_spec.shape}")
    print("You can SKIP Sections 10-11 now too.")
else:
    print("No cached spectrograms found yet.")

## 2. Download the IMS Bearing dataset

Source: NASA Prognostics Data Repository, hosted on a public S3 bucket
(more reliable for scripted downloads than data.nasa.gov's portal). This is
a **large download (~1-2 GB)** — expect this cell to take several minutes.

In [ ]:
import os

DATA_URL = "https://phm-datasets.s3.amazonaws.com/NASA/4.+Bearings.zip"
RAW_DIR = "ims_raw"
BEARINGS_DIR = os.path.join(RAW_DIR, "4. Bearings")
os.makedirs(RAW_DIR, exist_ok=True)
zip_path = "Bearings.zip"

TARGET_MARKER = os.path.join(BEARINGS_DIR, "1st_test")

if os.path.exists(TARGET_MARKER):
    print("Dataset already fully extracted, skipping everything.")
else:
    # Step 1: download the outer zip
    if not os.path.exists(zip_path):
        print("Downloading IMS Bearing dataset (~1 GB, this can take a few minutes)...")
        exit_code = os.system(f'wget -q -O "{zip_path}" "{DATA_URL}"')
        size_mb = os.path.getsize(zip_path) / 1e6 if os.path.exists(zip_path) else 0
        print(f"Download exit code: {exit_code} | size: {size_mb:.1f} MB")
        if size_mb < 100:
            print("WARNING: file looks too small — download may have failed. "
                  "Use the manual upload cell (2b) below instead.")
    else:
        print("Bearings.zip already present.")

    # Step 2: unzip the outer .zip -> reveals "4. Bearings/IMS.7z"
    if not os.path.exists(os.path.join(BEARINGS_DIR, "IMS.7z")):
        print("Unzipping outer archive...")
        os.system(f'unzip -oq "{zip_path}" -d {RAW_DIR}')

    # Step 3: extract the nested IMS.7z -> reveals 1st_test.rar, 2nd_test.rar, 3rd_test.rar
    seven_zip_path = os.path.join(BEARINGS_DIR, "IMS.7z")
    if os.path.exists(seven_zip_path) and not os.path.exists(os.path.join(BEARINGS_DIR, "1st_test.rar")):
        print("Installing p7zip and extracting IMS.7z (this takes a minute)...")
        os.system("apt-get install -y p7zip-full -qq > /dev/null 2>&1")
        os.system(f'7z x "{seven_zip_path}" -o"{BEARINGS_DIR}" -y > /dev/null')

    # Step 4: extract the .rar files -> reveals 1st_test/, 2nd_test/ (and a
    # third folder, oddly named "4th_test" in this repackaging — not needed
    # for this notebook, which targets 1st_test's Bearing 3 inner race defect)
    rar_files = [f for f in os.listdir(BEARINGS_DIR) if f.endswith(".rar")] if os.path.exists(BEARINGS_DIR) else []
    if rar_files and not os.path.exists(TARGET_MARKER):
        print("Installing unrar and extracting .rar archives...")
        os.system("apt-get install -y unrar -qq > /dev/null 2>&1")
        for rar_file in rar_files:
            rar_path = os.path.join(BEARINGS_DIR, rar_file)
            print(f"  Extracting {rar_file} ...")
            os.system(f'unrar x -y "{rar_path}" "{BEARINGS_DIR}/" > /dev/null')

    if os.path.exists(TARGET_MARKER):
        print("\nExtraction complete. 1st_test/ is ready.")
    else:
        print("\n1st_test/ still not found — see the manual upload fallback (2b) below,")
        print("or check the diagnostic cell further down to see what actually extracted.")


In [ ]:
# Diagnostic — run this if Section 2 above didn't find 1st_test/
if os.path.exists(BEARINGS_DIR):
    for item in sorted(os.listdir(BEARINGS_DIR)):
        full_path = os.path.join(BEARINGS_DIR, item)
        if os.path.isdir(full_path):
            print(f"{item}/  -> {os.listdir(full_path)[:5]}")
        else:
            print(item)
else:
    print(f"{BEARINGS_DIR} doesn't exist yet — the outer zip/7z extraction didn't run.")

### 2b. (Only if the download above failed) Manual upload fallback

Download `4. Bearings.zip` yourself from
[NASA's data repository listing](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/)
and upload it here if the automatic download didn't work.

In [ ]:
from google.colab import files

uploaded = files.upload()
for fname in uploaded:
    if fname.endswith(".zip"):
        !unzip -oq "{fname}" -d {RAW_DIR}
        print(f"Unzipped {fname}")

## 3. Explore the dataset structure

**Set 1** (`1st_test`): 8 channels (2 accelerometers per bearing, bearings 1-4),
20 kHz sampling, ~20,480 samples per file, one file every 10 minutes
(every 5 min for the first 43 files). Run ends with an **inner race defect
on Bearing 3** and a **roller element defect on Bearing 4** — we'll focus on
**Bearing 3, channel 5**, since inner race defects are a well-documented
fault mode in this dataset.

In [ ]:
import numpy as np
import pandas as pd

# Confirmed correct path after extraction: files sit directly inside
# "4. Bearings/1st_test/" (no further nesting, no txt/ subfolder for this set).
TEST_SET_DIR = os.path.join(BEARINGS_DIR, "1st_test")

if not os.path.exists(TEST_SET_DIR):
    raise FileNotFoundError(
        f"{TEST_SET_DIR} not found. Run the diagnostic cell in Section 2 "
        "to see what actually got extracted, and adjust TEST_SET_DIR to match."
    )

all_files = sorted(os.listdir(TEST_SET_DIR))
print(f"Found {len(all_files)} snapshot files in {TEST_SET_DIR}")
print("First few:", all_files[:3])
print("Last few:", all_files[-3:])

# Peek at one file's structure
sample = pd.read_csv(os.path.join(TEST_SET_DIR, all_files[0]), sep="\t", header=None)
print("\nSample file shape (rows=samples, cols=channels):", sample.shape)

## 4. Feature extraction (time-domain + frequency-domain)

In [ ]:
from scipy.stats import kurtosis, skew
from scipy.fft import rfft, rfftfreq

BEARING_CHANNEL = 4  # 0-indexed: channel 5 (Bearing 3, first accelerometer)
SAMPLING_RATE = 20000  # Hz


def extract_features(signal, sampling_rate=SAMPLING_RATE):
    """Time-domain + frequency-domain features for one vibration snapshot."""
    rms = np.sqrt(np.mean(signal ** 2))
    peak = np.max(np.abs(signal))
    crest_factor = peak / (rms + 1e-8)
    kurt = kurtosis(signal)
    skewness = skew(signal)
    std = np.std(signal)

    # FFT — dominant frequency and energy in a few bearing-fault-relevant bands
    n = len(signal)
    freqs = rfftfreq(n, d=1 / sampling_rate)
    magnitude = np.abs(rfft(signal))

    dominant_freq = freqs[np.argmax(magnitude)]
    total_energy = np.sum(magnitude ** 2)

    # Energy in low/mid/high frequency bands (rough bearing-fault heuristic bands)
    band_edges = [0, 1000, 3000, 6000, sampling_rate // 2]
    band_energies = []
    for i in range(len(band_edges) - 1):
        mask = (freqs >= band_edges[i]) & (freqs < band_edges[i + 1])
        band_energies.append(np.sum(magnitude[mask] ** 2) / (total_energy + 1e-8))

    return {
        "rms": rms, "peak": peak, "crest_factor": crest_factor,
        "kurtosis": kurt, "skewness": skewness, "std": std,
        "dominant_freq": dominant_freq,
        "band_energy_0_1k": band_energies[0],
        "band_energy_1k_3k": band_energies[1],
        "band_energy_3k_6k": band_energies[2],
        "band_energy_6k_plus": band_energies[3],
    }

## 5. Build the feature dataset across all snapshots

This reads every file in the run (thousands of files, ~20k samples each), so
it can take a few minutes. Set `SUBSAMPLE_EVERY_N` higher to speed this up
for a quick test run, then set it back to 1 for the full dataset.

In [ ]:
SUBSAMPLE_EVERY_N = 1  # use every Nth file; set higher (e.g. 5) to run faster

records = []
selected_files = all_files[::SUBSAMPLE_EVERY_N]

for i, fname in enumerate(selected_files):
    filepath = os.path.join(TEST_SET_DIR, fname)
    df = pd.read_csv(filepath, sep="\t", header=None)
    signal = df[BEARING_CHANNEL].values

    feats = extract_features(signal)
    feats["filename"] = fname
    feats["file_index"] = i
    records.append(feats)

    if (i + 1) % 200 == 0:
        print(f"  Processed {i+1}/{len(selected_files)} files...")

feature_df = pd.DataFrame(records)
print(f"\nBuilt feature dataset: {feature_df.shape}")
feature_df.head()

### Save processed features to Drive (for next time)

Run this once after Section 5 completes — future sessions can load this
instead of re-parsing all 2,156 raw files.

In [ ]:
feature_df.to_csv(CACHE_FEATURE_DF, index=False)
print(f"Saved feature_df to {CACHE_FEATURE_DF} ({feature_df.shape})")

## 6. Label health stages

The IMS dataset doesn't come with ground-truth labels for *when* degradation
began — only that failure occurred at the very end of the run. Following
common practice in the literature, we split the run into three stages by
**relative time position**: the first 60% of the run is labeled Normal, the
next 25% Degrading, and the final 15% Near-Failure. This is a heuristic, not
a physically verified transition point — worth stating explicitly in any
write-up rather than implying it's ground truth.

In [ ]:
# REVISED APPROACH: the original 60/25/15% time-based split assumed
# degradation begins at a fixed fraction of the run's duration. Plotting the
# actual RMS trend showed this was wrong for this bearing — RMS stays flat
# until roughly 90%+ of the way through the run, meaning most of what the
# old heuristic called "Degrading" was statistically indistinguishable from
# "Normal." That's very likely why both the Random Forest and CNN struggled
# on the Normal/Degrading boundary regardless of class weighting — the
# labels themselves didn't reflect a real distinction in much of the data.
#
# Fix: derive the transition points from the RMS signal itself. Use the
# early part of the run (first 20%, assumed healthy/run-in-complete) as a
# baseline, then flag "Degrading" where RMS first rises meaningfully above
# that baseline, and "Near-Failure" where it rises much further. A rolling
# mean smooths out single-snapshot noise spikes so a brief blip doesn't
# falsely trigger an early transition.

BASELINE_FRACTION = 0.20      # first 20% of the run defines "healthy" baseline
DEGRADING_N_STD = 3.0         # degrading: RMS > baseline_mean + 3 std, sustained
NEAR_FAILURE_N_STD = 8.0      # near-failure: RMS > baseline_mean + 8 std, sustained
SMOOTHING_WINDOW = 10         # rolling window to filter out single-snapshot noise
DEGRADING_SUSTAIN = 20        # degrading onset should hold for a while (gradual wear)
NEAR_FAILURE_SUSTAIN = 5      # catastrophic failure is often a short, sharp spike —
                               # requiring 20 sustained snapshots was too strict and
                               # produced ZERO Near-Failure labels on real data, since
                               # the actual failure spike only lasted a handful of
                               # snapshots (visible as a brief tail spike in the RMS plot)

n_files = len(feature_df)
baseline_n = max(int(n_files * BASELINE_FRACTION), 10)

baseline_rms = feature_df["rms"].iloc[:baseline_n]
baseline_mean = baseline_rms.mean()
baseline_std = baseline_rms.std()

rms_smoothed = feature_df["rms"].rolling(SMOOTHING_WINDOW, min_periods=1, center=True).mean()

degrading_threshold = baseline_mean + DEGRADING_N_STD * baseline_std
near_failure_threshold = baseline_mean + NEAR_FAILURE_N_STD * baseline_std

# Find the FIRST index where the smoothed RMS crosses each threshold and
# STAYS above it for `sustain_for` snapshots (avoids a single noisy blip
# triggering a false-early transition, while still allowing short but real
# failure spikes to register — hence the shorter near-failure window).
def first_sustained_crossing(series, threshold, sustain_for):
    above = series >= threshold
    for i in range(len(above) - sustain_for):
        if above.iloc[i:i + sustain_for].all():
            return i
    return len(series)  # never sustained — no crossing found


degrading_cutoff = first_sustained_crossing(rms_smoothed, degrading_threshold, DEGRADING_SUSTAIN)
near_failure_cutoff = first_sustained_crossing(rms_smoothed, near_failure_threshold, NEAR_FAILURE_SUSTAIN)
near_failure_cutoff = max(near_failure_cutoff, degrading_cutoff)  # keep ordering sane

if near_failure_cutoff == len(feature_df):
    print("WARNING: no sustained Near-Failure crossing found even with the shorter "
          "window. This bearing's data may end before a clear catastrophic failure "
          "signature, or NEAR_FAILURE_N_STD may still be too high — consider lowering "
          "it (e.g. to 5.0) and re-running this cell.")


def assign_label(idx):
    if idx < degrading_cutoff:
        return "Normal"
    elif idx < near_failure_cutoff:
        return "Degrading"
    else:
        return "Near-Failure"


feature_df["label"] = feature_df["file_index"].apply(assign_label)

print(f"Baseline RMS: mean={baseline_mean:.4f}, std={baseline_std:.4f}")
print(f"Degrading threshold: {degrading_threshold:.4f}  -> first crossing at index {degrading_cutoff}")
print(f"Near-Failure threshold: {near_failure_threshold:.4f}  -> first crossing at index {near_failure_cutoff}")
print(f"\n(For comparison, the old heuristic used fixed cutoffs at "
      f"{int(n_files*0.60)} and {int(n_files*0.85)} regardless of the data.)")
print()
print(feature_df["label"].value_counts())

## 7. Visualize RMS trend and label boundaries

In [ ]:
import matplotlib.pyplot as plt

colors = {"Normal": "#3DDC84", "Degrading": "#F5A623", "Near-Failure": "#FF5A5A"}

fig, ax = plt.subplots(figsize=(12, 4))
for label, group in feature_df.groupby("label"):
    ax.scatter(group["file_index"], group["rms"], s=6, color=colors[label], label=label)

ax.set_xlabel("Snapshot index (time)")
ax.set_ylabel("RMS vibration amplitude")
ax.set_title("Bearing 3 — RMS Trend Over the Run-to-Failure Test")
ax.legend()
plt.tight_layout()
plt.show()

print("\nIf RMS visibly spikes well before the 'Near-Failure' cutoff, that's a")
print("sign the heuristic split doesn't perfectly match the true degradation")
print("onset — worth noting honestly rather than adjusting labels to fit.")

## 8. Random Forest classifier on hand-engineered features

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# IMPORTANT — data leakage fix: labels (Normal/Degrading/Near-Failure) were
# created in Section 6 by thresholding on RMS. Including RMS (or features
# that are essentially just RMS in different units — peak, crest_factor,
# std) as a classifier INPUT makes this circular: the model can trivially
# learn "predict the RMS-based label using RMS," which produces artificially
# perfect scores (this is exactly why the earlier threshold-tuning run hit
# threshold=1.000 with perfect precision/recall — not genuine skill).
#
# Excluding amplitude-derived features forces the model to rely on genuinely
# independent signal: the SHAPE of the vibration spectrum, not its overall
# amplitude. This is a fair, non-circular test of whether spectral shape
# alone carries real predictive information about bearing health.
FEATURE_COLS = [
    "kurtosis", "skewness", "dominant_freq",
    "band_energy_0_1k", "band_energy_1k_3k",
    "band_energy_3k_6k", "band_energy_6k_plus",
]
# Excluded (leaky, essentially redundant with RMS): "rms", "peak", "crest_factor", "std"

# NOTE: chronological split, not random shuffle — random shuffling would leak
# future degradation information into training, since consecutive snapshots
# are highly correlated in a run-to-failure sequence.
#
# IMPORTANT: a single GLOBAL 80/20 cutoff would put the entire test set
# inside the Degrading/Near-Failure region (since those are the last 40% of
# the run) — the model would never even be evaluated on "Normal" snapshots.
# Instead, split chronologically WITHIN each class: the earliest 80% of each
# class's snapshots go to train, the latest 20% go to test. This keeps time
# order intact (no shuffling within a class) while guaranteeing all three
# classes appear in both train and test.
train_idx_list, test_idx_list = [], []
for label in ["Normal", "Degrading", "Near-Failure"]:
    label_indices = feature_df.index[feature_df["label"] == label].tolist()
    cutoff = int(len(label_indices) * 0.8)
    train_idx_list.extend(label_indices[:cutoff])
    test_idx_list.extend(label_indices[cutoff:])

train_idx = sorted(train_idx_list)
test_idx = sorted(test_idx_list)

X_train = feature_df.loc[train_idx, FEATURE_COLS]
X_test = feature_df.loc[test_idx, FEATURE_COLS]
y_train = feature_df.loc[train_idx, "label"]
y_test = feature_df.loc[test_idx, "label"]

print("Train class distribution:\n", y_train.value_counts())
print("\nTest class distribution:\n", y_test.value_counts())

# NOTE: class_weight="balanced" was too aggressive here. With ~1694 Normal
# vs. only 15 Degrading training examples, "balanced" weights each Degrading
# example ~113x more than Normal — extreme enough that the model became
# trigger-happy about predicting Degrading, producing 0.01 precision on it
# (i.e. almost every Degrading prediction was actually a misclassified
# Normal). A capped, milder weighting keeps some correction for the
# imbalance without letting the rarest class dominate the decision boundary.
CLASS_WEIGHTS = {"Normal": 1, "Degrading": 8, "Near-Failure": 8}

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight=CLASS_WEIGHTS)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
print(classification_report(y_test, rf_preds))

cm = confusion_matrix(y_test, rf_preds, labels=["Normal", "Degrading", "Near-Failure"])
disp = ConfusionMatrixDisplay(cm, display_labels=["Normal", "Degrading", "Near-Failure"])
disp.plot(cmap="Blues")
plt.title("Random Forest — Confusion Matrix (capped class weights)")
plt.show()

## 8b. Supplementary: binary classification (Normal vs. Anomalous)

With only 15-19 real examples each of Degrading and Near-Failure, reliable
3-class separation is a genuinely hard, data-starved problem — worth
reporting honestly rather than papering over. As a complementary, more
achievable framing: collapse Degrading + Near-Failure into a single
"Anomalous" class and check whether the model can at least reliably flag
*something is wrong* vs. *everything is fine* — a simpler but still
practically useful predictive-maintenance signal (e.g., "schedule an
inspection" vs. "no action needed").

In [ ]:
y_train_binary = y_train.apply(lambda l: "Normal" if l == "Normal" else "Anomalous")
y_test_binary = y_test.apply(lambda l: "Normal" if l == "Normal" else "Anomalous")

print("Train distribution:\n", y_train_binary.value_counts())
print("\nTest distribution:\n", y_test_binary.value_counts())

# class_weight="balanced" was still too aggressive here too (1694:30 is a
# ~56x weight ratio) — it produced 100% recall but 6% precision on Anomalous,
# meaning ~94% of "Anomalous" alerts were false alarms. Rather than continuing
# to guess at weight values, switch to a more principled fix: train WITHOUT
# extreme weighting, then tune the DECISION THRESHOLD on predicted
# probabilities to find a defensible precision/recall balance. This is the
# standard approach for imbalanced classification with a probabilistic model.
from sklearn.metrics import precision_recall_curve, f1_score

rf_binary = RandomForestClassifier(
    n_estimators=200, max_depth=8, random_state=42, class_weight=None  # no weighting
)
rf_binary.fit(X_train, y_train_binary)

# Probability of the "Anomalous" class for each test sample
anomalous_class_idx = list(rf_binary.classes_).index("Anomalous")
y_proba = rf_binary.predict_proba(X_test)[:, anomalous_class_idx]
y_test_binary_numeric = (y_test_binary == "Anomalous").astype(int)

# IMPORTANT — second methodological issue found: picking the threshold that
# maximizes F1 by searching over the TEST set's own precision-recall curve is
# a form of test-set leakage. With only 8 positive test examples, this search
# can effectively find a threshold that perfectly separates those exact 8
# points by coincidence, producing an artificially perfect score that has
# nothing to do with real generalization — which is exactly what happened
# (threshold landed at a suspicious extreme with 100% precision/recall).
#
# Honest fix: report performance at the STANDARD default threshold (0.5),
# chosen without looking at test labels at all. This is a lower, less
# flattering number — but a trustworthy one.
default_threshold = 0.5
rf_binary_preds_default = np.where(y_proba >= default_threshold, "Anomalous", "Normal")

print(f"=== Default threshold ({default_threshold}) — the honest, reportable result ===")
print(classification_report(y_test_binary, rf_binary_preds_default))

cm_binary = confusion_matrix(y_test_binary, rf_binary_preds_default, labels=["Normal", "Anomalous"])
disp = ConfusionMatrixDisplay(cm_binary, display_labels=["Normal", "Anomalous"])
disp.plot(cmap="Greens")
plt.title(f"Random Forest — Binary Classification (default threshold={default_threshold})")
plt.show()

# The precision-recall curve is still useful to LOOK AT (it shows the
# precision/recall tradeoff shape across all possible thresholds), but we do
# NOT pick a threshold from it and report that as if it were a fair test
# result — that would repeat the leakage. Shown here for exploratory
# diagnostic purposes only, clearly labeled as such.
precisions, recalls, thresholds = precision_recall_curve(y_test_binary_numeric, y_proba)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recalls, precisions, color="#4C72B0")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve (EXPLORATORY ONLY — not used to pick a reported threshold)")
plt.tight_layout()
plt.show()

print("\nNote: with only 30 real anomalous training / 8 test examples, any threshold")
print("tuned on this test set would be overfit to those exact points. The default")
print("0.5 threshold above is the fair, honest number to report. A properly tuned")
print("threshold would need a separate validation set — not feasible to carve out")
print("responsibly from this few examples without leaving test/train too sparse.")

## 9. Feature importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values()

fig, ax = plt.subplots(figsize=(7, 5))
importances.plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Random Forest Feature Importance")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

## 10. Generate spectrograms for the CNN approach

Each raw vibration snapshot (~1 second at 20 kHz) gets converted into a
spectrogram — a 2D time-frequency image — which the CNN treats like any
other image classification problem.

In [ ]:
from scipy.signal import spectrogram as scipy_spectrogram

SPEC_SIZE = 64  # resize spectrograms to SPEC_SIZE x SPEC_SIZE for the CNN


def compute_spectrogram(signal, sampling_rate=SAMPLING_RATE, out_size=SPEC_SIZE):
    f, t, Sxx = scipy_spectrogram(signal, fs=sampling_rate, nperseg=256, noverlap=128)
    Sxx_db = 10 * np.log10(Sxx + 1e-10)

    # Resize to a fixed size via simple interpolation (keeps things lightweight —
    # avoids pulling in a heavier image-resize dependency for this step)
    from scipy.ndimage import zoom
    zoom_factors = (out_size / Sxx_db.shape[0], out_size / Sxx_db.shape[1])
    resized = zoom(Sxx_db, zoom_factors)

    # Normalize to 0-1
    resized = (resized - resized.min()) / (resized.max() - resized.min() + 1e-8)
    return resized.astype(np.float32)


# Quick visual sanity check: one spectrogram per class
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, label in zip(axes, ["Normal", "Degrading", "Near-Failure"]):
    idx = feature_df[feature_df["label"] == label]["file_index"].iloc[0]
    fname = feature_df[feature_df["file_index"] == idx]["filename"].iloc[0]
    sig = pd.read_csv(os.path.join(TEST_SET_DIR, fname), sep="\t", header=None)[BEARING_CHANNEL].values
    spec = compute_spectrogram(sig)
    ax.imshow(spec, aspect="auto", origin="lower", cmap="viridis")
    ax.set_title(label)
plt.tight_layout()
plt.show()

## 11. Build the full spectrogram dataset

Reuses the same files already read in Section 5 — recomputing spectrograms
instead of re-reading from disk where possible would be faster, but reading
fresh keeps this section runnable independently if you skip Sections 5-9.

In [ ]:
spectrograms = []
for i, fname in enumerate(feature_df["filename"]):
    filepath = os.path.join(TEST_SET_DIR, fname)
    sig = pd.read_csv(filepath, sep="\t", header=None)[BEARING_CHANNEL].values
    spectrograms.append(compute_spectrogram(sig))
    if (i + 1) % 200 == 0:
        print(f"  Computed {i+1}/{len(feature_df)} spectrograms...")

X_spec = np.array(spectrograms, dtype=np.float32)
print("Spectrogram dataset shape:", X_spec.shape)

### Save spectrograms to Drive (for next time)

Run this once after Section 11 completes — this is the slower of the two
caching steps to regenerate (reads every raw file again), so caching it
saves the most time on future runs.

In [ ]:
np.save(CACHE_SPECTROGRAMS, X_spec)
print(f"Saved spectrograms to {CACHE_SPECTROGRAMS} ({X_spec.shape})")

## 12. CNN classifier on spectrograms

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

label_encoder = LabelEncoder()
label_encoder.fit(["Normal", "Degrading", "Near-Failure"])
y_encoded = label_encoder.transform(feature_df["label"])

# Same per-class chronological split as the Random Forest (Section 8), for a
# fair comparison — train_idx/test_idx are row positions into feature_df,
# which line up with X_spec since spectrograms were built in the same order.
X_spec_train, X_spec_test = X_spec[train_idx], X_spec[test_idx]
y_spec_train, y_spec_test = y_encoded[train_idx], y_encoded[test_idx]


class SpectrogramDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # add channel dim
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(SpectrogramDataset(X_spec_train, y_spec_train), batch_size=32, shuffle=True)
test_loader = DataLoader(SpectrogramDataset(X_spec_test, y_spec_test), batch_size=32, shuffle=False)


class SpectrogramCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        # SPEC_SIZE=64 -> after 3x maxpool(2): 64/8=8, so 64 channels * 8 * 8
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.fc(self.conv(x))


# Class weighting — full "balanced" weighting produced a ~113x ratio
# (mirroring the exact same problem found in the Random Forest earlier),
# which caused the training loss to bounce around instead of converging: a
# single batch containing one of the 15 rare-class examples would suddenly
# contribute 38x the gradient, causing large, unstable updates. Using capped
# weights instead — same fix philosophy as the RF — keeps some correction
# for the imbalance without letting a handful of examples destabilize
# training. We do NOT pick a "best epoch" based on test-set performance
# during training, since that would reintroduce the same test-set leakage
# already found and fixed twice in this notebook — we report whatever the
# final epoch gives us, honestly, same principle as Section 8b's default
# threshold.
CLASS_WEIGHTS_CNN = {"Normal": 1.0, "Degrading": 8.0, "Near-Failure": 8.0}

class_labels = np.unique(y_spec_train)
class_names_ordered = label_encoder.inverse_transform(class_labels)
class_weights = np.array([CLASS_WEIGHTS_CNN[name] for name in class_names_ordered])
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights (capped, by encoded label order):", dict(zip(class_names_ordered, class_weights)))

cnn = SpectrogramCNN().to(device)
optimizer = torch.optim.Adam(cnn.parameters(), lr=5e-4)  # slightly lower LR for added stability
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

EPOCHS = 20
for epoch in range(EPOCHS):
    cnn.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = cnn(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(cnn.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  train_loss={total_loss/len(train_loader.dataset):.3f}")

## 13. Evaluate the CNN and compare to Random Forest

In [ ]:
cnn.eval()
cnn_preds_encoded = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = cnn(X_batch).argmax(dim=1).cpu().numpy()
        cnn_preds_encoded.extend(preds)

cnn_preds = label_encoder.inverse_transform(cnn_preds_encoded)
y_test_labels = label_encoder.inverse_transform(y_spec_test)

print("=== CNN (Spectrogram) ===")
print(classification_report(y_test_labels, cnn_preds))

cm_cnn = confusion_matrix(y_test_labels, cnn_preds, labels=["Normal", "Degrading", "Near-Failure"])
disp = ConfusionMatrixDisplay(cm_cnn, display_labels=["Normal", "Degrading", "Near-Failure"])
disp.plot(cmap="Oranges")
plt.title("CNN (Spectrogram) — Confusion Matrix")
plt.show()

from sklearn.metrics import accuracy_score, f1_score
rf_acc = accuracy_score(y_test, rf_preds)
cnn_acc = accuracy_score(y_test_labels, cnn_preds)
rf_f1 = f1_score(y_test, rf_preds, average="macro")
cnn_f1 = f1_score(y_test_labels, cnn_preds, average="macro")

comparison = pd.DataFrame([
    {"Model": "Random Forest (features)", "Accuracy": round(rf_acc, 3), "Macro F1": round(rf_f1, 3)},
    {"Model": "CNN (spectrogram)", "Accuracy": round(cnn_acc, 3), "Macro F1": round(cnn_f1, 3)},
])
comparison

## Notes

- **Why chronological split, not random**: consecutive snapshots in a
  run-to-failure test are highly autocorrelated. A random shuffle would put
  near-identical neighboring snapshots in both train and test, inflating
  accuracy artificially. The chronological split (train on the earlier part
  of the run, test on the later part) is the honest evaluation here — and
  arguably harder, since it's genuinely predicting unseen degradation stages.
- **Label caveat**: the Normal/Degrading/Near-Failure split is a time-based
  heuristic (60/25/15%), not verified ground truth. State this explicitly in
  any write-up — a reviewer familiar with this dataset will expect that
  caveat and will trust the work more for including it.
- **To extend**: try Bearing 4 (roller defect) or Set 2/3 for other fault
  types, or add more frequency bands tied to actual bearing fault frequencies
  (BPFO/BPFI/BSF, computable from bearing geometry) instead of the generic
  bands used here.
- **To speed up experimentation**: raise `SUBSAMPLE_EVERY_N` in Section 5 to
  process every Nth file while you're iterating, then set it back to 1 for
  final results.

---
# Stage 3 Dashboard — Export Artifacts

Saves everything the Flask dashboard needs: the leak-free Random Forest
model, the CNN, feature columns, and a set of demo snapshots (spanning all
three health states) with their spectrograms and true labels — so the
dashboard can demonstrate real predictions without needing the full raw
dataset re-downloaded.

## 14. Install export dependency

In [ ]:
!pip install -q joblib

## 15. Save all artifacts

In [ ]:
import os
import json as json_lib
import joblib

ARTIFACT_DIR = "stage3_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# 1. Leak-free Random Forest model (3-class, from Section 8) + its feature columns
joblib.dump(rf, os.path.join(ARTIFACT_DIR, "rf_model.pkl"))
with open(os.path.join(ARTIFACT_DIR, "feature_cols.json"), "w") as f:
    json_lib.dump(FEATURE_COLS, f)

# 2. CNN model (from Section 12) + its config, so the dashboard can rebuild
#    the exact same architecture before loading weights
torch.save(cnn.state_dict(), os.path.join(ARTIFACT_DIR, "cnn_model.pt"))
with open(os.path.join(ARTIFACT_DIR, "cnn_config.json"), "w") as f:
    json_lib.dump({
        "spec_size": SPEC_SIZE,
        "classes": list(label_encoder.classes_),
    }, f)

# 3. Demo snapshots — pick a handful spanning all three classes, so the
#    dashboard has real examples to run predictions on. Includes both the
#    hand-engineered features (for the RF) and the spectrogram (for the CNN).
#
# IMPORTANT: sample ONLY from the held-out TEST set (test_idx), not the full
# feature_df. Sampling from the full dataset would include training examples
# the models have already seen — a model "correctly" predicting a training
# example it memorized is not a real demonstration of its ability, and would
# directly contradict the honestly-measured test performance (e.g. RF's
# 0.00 precision/recall on Degrading came from test_idx rows specifically —
# showing a training-set Degrading example instead would misleadingly look
# far better than the model's real, reported performance).
demo_records = []
demo_spectrograms = []

test_feature_df = feature_df.loc[test_idx]

rng = np.random.RandomState(42)
for label in ["Normal", "Degrading", "Near-Failure"]:
    label_rows = test_feature_df[test_feature_df["label"] == label]
    n_pick = min(6, len(label_rows))
    picked = label_rows.sample(n=n_pick, random_state=42) if len(label_rows) > 0 else label_rows
    for _, row in picked.iterrows():
        demo_records.append(row)
        sig = pd.read_csv(os.path.join(TEST_SET_DIR, row["filename"]), sep="\t", header=None)[BEARING_CHANNEL].values
        demo_spectrograms.append(compute_spectrogram(sig))

print(f"Sampled demo snapshots exclusively from the held-out test set "
      f"({len(test_idx)} test rows available across all classes).")

demo_df = pd.DataFrame(demo_records).reset_index(drop=True)
demo_df.to_csv(os.path.join(ARTIFACT_DIR, "demo_snapshots.csv"), index=False)
np.save(os.path.join(ARTIFACT_DIR, "demo_spectrograms.npy"), np.array(demo_spectrograms, dtype=np.float32))

# 4. Known model limitations, saved as text so the dashboard can display
#    them honestly rather than implying more reliability than is warranted
limitations = {
    "rf_3class": "Degrading: 0.00 precision/recall (undetectable with leak-free features). "
                 "Near-Failure: 0.43 precision, 0.75 recall (real but modest signal via kurtosis).",
    "cnn_3class": "Failed to learn any signal for Degrading or Near-Failure (0.00/0.00 both) — "
                  "effectively always predicts Normal. Only 15 training examples per rare class.",
    "sample_size": "Only 30 total anomalous snapshots in the entire run — both models' rare-class "
                   "metrics are statistically noisy given so few examples.",
}
with open(os.path.join(ARTIFACT_DIR, "limitations.json"), "w") as f:
    json_lib.dump(limitations, f)

print("Saved artifacts:")
for fname in sorted(os.listdir(ARTIFACT_DIR)):
    size_kb = os.path.getsize(os.path.join(ARTIFACT_DIR, fname)) / 1024
    print(f"  {fname:30s} {size_kb:8.1f} KB")
print(f"\nDemo snapshots: {len(demo_df)} total, spanning {demo_df['label'].nunique()} classes")
print(demo_df["label"].value_counts())

## 16. Zip and download artifacts

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("stage3_artifacts", "zip", ARTIFACT_DIR)
print("Created stage3_artifacts.zip — downloading...")
files.download("stage3_artifacts.zip")
print("\nExtract this into the 'artifacts/' folder of the Stage 3 Flask dashboard, then run: python app.py")